In [15]:
import pandas as pd
import sqlite3
import os

# --- PHASE 1: INITIALISIERUNG (Einmalig) ---
# Wir prüfen, ob die DB schon existiert, um Zeit zu sparen
if not os.path.exists('apple_apps.db'):
    print("Initialisiere Datenbank aus CSV... (das dauert einen Moment)")
    
    # 1. Rohdaten laden
    df = pd.read_csv('appleAppData.csv')
    
    # 2. Preprocessing (Deine Logik von vorhin)
    # Nur bewertete Apps und Typ-Kategorisierung
    dashboard_df = df[df['Average_User_Rating'] > 0].copy()
    dashboard_df['Type'] = dashboard_df['Free'].apply(lambda x: 'Free' if x else 'Paid')
    
    # 3. In SQL speichern
    conn = sqlite3.connect('apple_apps.db')
    dashboard_df.to_sql('apps', conn, if_exists='replace', index=False)
    conn.close()
    print("Datenbank 'apple_apps.db' erfolgreich erstellt.")
else:
    print("Datenbank bereits vorhanden. Springe direkt zur Analyse.")

# --- PHASE 2: AB JETZT NUR NOCH SQL ---
def get_data(query):
    conn = sqlite3.connect('apple_apps.db')
    result = pd.read_sql(query, conn)
    conn.close()
    return result

# Beispiel: Den "neuen" dashboard_df direkt aus der DB ziehen
dashboard_df = get_data("SELECT * FROM apps")
print(f"Dataset ready from SQL with {len(dashboard_df)} rated apps.")

Datenbank bereits vorhanden. Springe direkt zur Analyse.
Dataset ready from SQL with 546056 rated apps.


In [19]:
# 1. Aggregation direkt per SQL-Abfrage
# Wir berechnen Durchschnitt und Anzahl direkt in der Datenbank
query_treemap = """
SELECT 
    Primary_Genre, 
    AVG(Average_User_Rating) as Average_Rating, 
    COUNT(App_Name) as App_Count
FROM apps
GROUP BY Primary_Genre
"""

genre_stats = get_data(query_treemap)

# 2. Die Visualisierung bleibt fast identisch (nur die Spaltennamen nutzen)
fig = px.treemap(
    genre_stats, 
    path=['Primary_Genre'], 
    values='App_Count',
    color='Average_Rating',
    color_continuous_scale='RdYlGn', 
    title='Global App Store: Genre Popularity vs. User Satisfaction (SQL Aggregated)'
)

fig.update_layout(
    margin=dict(t=50, l=25, r=25, b=25),
    template='plotly_dark' # Beibehalten des Dark-Themes für Konsistenz
)
fig.show()

print(f"Treemap generiert aus {len(genre_stats)} Genre-Aggregaten.")

Treemap generiert aus 26 Genre-Aggregaten.


In [20]:
# 1. SQL-Abfrage für das Premium-Segment
# Wir filtern auf Price > 0 und aggregieren direkt in der DB
query_price = """
SELECT 
    Primary_Genre, 
    AVG(Price) as Avg_Price, 
    COUNT(App_Name) as App_Count
FROM apps
WHERE Price > 0
GROUP BY Primary_Genre
ORDER BY Avg_Price DESC
LIMIT 15
"""

price_analysis = get_data(query_price)

# 2. Visualisierung
fig_price = px.bar(
    price_analysis, 
    x='Avg_Price', 
    y='Primary_Genre', 
    orientation='h',
    color='Avg_Price',
    color_continuous_scale='Blues',
    labels={'Avg_Price': 'Average Price ($)', 'Primary_Genre': 'Category'},
    title='Top 15 Most Expensive App Categories (Premium Segment - SQL Aggregated)'
)

fig_price.update_layout(
    yaxis={'categoryorder':'total ascending'},
    template='plotly_dark'
)
fig_price.show()

print(f"Preisanalyse für {len(price_analysis)} Top-Genres abgeschlossen.")

Preisanalyse für 15 Top-Genres abgeschlossen.


In [21]:
import plotly.graph_objects as go

# 1. SQL-Abfrage für die Top-Performer
# Wir sortieren erst nach Rating, dann nach Volumen
query_top = """
SELECT App_Name, Primary_Genre, Average_User_Rating, Reviews, Price
FROM apps
WHERE Reviews > 10000
ORDER BY Average_User_Rating DESC, Reviews DESC
LIMIT 10
"""

top_performers = get_data(query_top)

# 2. Schöne Darstellung als interaktive Plotly-Tabelle
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>App Name</b>', '<b>Genre</b>', '<b>Rating</b>', '<b>Reviews</b>', '<b>Price</b>'],
        fill_color='royalblue',
        align='left',
        font=dict(color='white', size=12)
    ),
    cells=dict(
        values=[
            top_performers.App_Name, 
            top_performers.Primary_Genre, 
            top_performers.Average_User_Rating, 
            top_performers.Reviews, 
            top_performers.Price
        ],
        fill_color='lavender',
        align='left',
        font=dict(color='black', size=11)
    ))
])

fig_table.update_layout(
    title='Global Top 10 Apps (High Rating & High Volume)',
    margin=dict(l=20, r=20, t=60, b=20)
)

fig_table.show()

In [22]:
# 1. SQL-Abfrage: Wir holen nur die Top-Apps nach Review-Volumen vor, 
# um den Rechenaufwand minimal zu halten.
query_weighted = """
SELECT App_Name, Primary_Genre, Average_User_Rating, Reviews
FROM apps
WHERE Reviews > 1000
ORDER BY Reviews DESC
LIMIT 500
"""

top_raw = get_data(query_weighted)

# 2. Berechnung des Weighted Scores in Python (da flexibler für komplexe Formeln)
top_raw['Weighted_Score'] = top_raw['Average_User_Rating'] * np.log10(top_raw['Reviews'] + 1)

# 3. Sortieren und die Top 20 für den Plot nehmen
top_weighted = top_raw.sort_values(by='Weighted_Score', ascending=False).head(20)

# 4. Visualisierung
fig_weighted = px.bar(
    top_weighted, 
    x='Weighted_Score', 
    y='App_Name', 
    color='Primary_Genre',
    orientation='h',
    title='The True Giants: Highest Impact Apps (Rating weighted by Volume)',
    labels={'Weighted_Score': 'Impact Score (Rating * log10(Reviews))'},
    template='plotly_dark'
)

fig_weighted.update_layout(yaxis={'categoryorder':'total ascending'})
fig_weighted.show()

In [25]:
# 1. SQL-Abfrage: Wir laden nur die numerischen Spalten, die wir wirklich brauchen
query_corr = """
SELECT Average_User_Rating, Reviews, Price, Size_MB
FROM apps
"""
df_numeric = get_data(query_corr)

# 2. Den Weighted_Score für die Korrelationsmatrix in Python ergänzen
# Wir nutzen dieselbe Logik wie im vorherigen Block
df_numeric['Weighted_Score'] = df_numeric['Average_User_Rating'] * np.log10(df_numeric['Reviews'] + 1)

# 3. Korrelationsmatrix berechnen
corr_matrix = df_numeric.corr()

# 4. Visualisierung (Heatmap)
fig_corr = px.imshow(
    corr_matrix,
    text_auto=".2f", # Begrenzung auf 2 Nachkommastellen für die Sauberkeit
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title="Correlation Matrix: What actually drives App Success? (SQL-Source)",
    template='plotly_dark'
)

fig_corr.show()

# 1. Korrelationen extrahieren
# Wir schauen uns speziell an, wie alles mit dem 'Weighted_Score' (Erfolg) zusammenhängt
correlations = corr_matrix['Weighted_Score'].sort_values(ascending=False)

# 2. Text-Ausgabe generieren
print("--- AUTOMATISIERTE KORRELATIONS-ANALYSE ---")
for index, value in correlations.items():
    if index == 'Weighted_Score':
        continue
    
    # Interpretation der Stärke
    strength = "stark" if abs(value) > 0.5 else "moderat" if abs(value) > 0.3 else "schwach"
    direction = "positiven" if value > 0 else "negativen"
    
    print(f"-> {index} hat einen {direction}, {strength}en Einfluss ({value:.2f}) auf den Erfolgsscore.")


--- AUTOMATISIERTE KORRELATIONS-ANALYSE ---
-> Average_User_Rating hat einen positiven, schwachen Einfluss (0.29) auf den Erfolgsscore.
-> Size_MB hat einen positiven, schwachen Einfluss (0.18) auf den Erfolgsscore.
-> Reviews hat einen positiven, schwachen Einfluss (0.14) auf den Erfolgsscore.
-> Price hat einen negativen, schwachen Einfluss (-0.00) auf den Erfolgsscore.


Die Analyse räumt mit dem Mythos auf, dass Erfolg im App Store käuflich oder durch reine Masse (Reviews) garantiert ist. Da selbst das Rating nur einen moderaten Einfluss hat, wird klar: Der wahre 'Impact' entsteht durch ein komplexes Zusammenspiel, bei dem die Qualität (Rating) die Basis bildet, aber erst durch die Reichweite (Reviews) skaliert wird

In [28]:
# 1. SQL-Abfrage: Filtern direkt in der Datenbank
# Wir laden nur Apps mit > 5000 Reviews und stellen sicher, dass Rating und Size existieren
query_scatter = """
SELECT App_Name, Reviews, Average_User_Rating, Size_MB, Primary_Genre
FROM apps
WHERE Reviews > 5000 
  AND Average_User_Rating IS NOT NULL 
  AND Size_MB IS NOT NULL
"""

heavy_hitters = get_data(query_scatter)

# 2. Den Scatter Plot erstellen
fig_scatter = px.scatter(
    heavy_hitters, 
    x="Reviews", 
    y="Average_User_Rating",
    size="Size_MB", 
    color="Primary_Genre",
    hover_name="App_Name",
    log_x=True, 
    title="High-Volume Apps: Rating vs. Reviews (Bubble Size = App Size)",
    size_max=40 
)

fig_scatter.update_layout(
    template='plotly_dark',
    xaxis_title="Number of Reviews (Log Scale)",
    yaxis_title="Average User Rating"
)

fig_scatter.show()

print(f"Plotting {len(heavy_hitters)} apps directly from SQL.")

Plotting 10561 apps directly from SQL.
